# Karar Ağaçları ile Sınıflandırma - Bank Customer Churn

**Veri Seti:** Bank Customer Churn Dataset (10.000 müşteri kaydı)

**Amaç:** Bir banka müşterisinin bankadan ayrılıp ayrılmayacağını (churn) tahmin etmek.

**Hedef Değişken:** `Exited` (1 = Ayrıldı, 0 = Kaldı)

**İyileştirmeler:**
- One-Hot Encoding (nominal değişkenler için)
- StandardScaler (KNN için ölçeklendirme)
- Cross-Validation (5-fold)
- Hiperparametre optimizasyonu (GridSearchCV)
- Dengesiz sınıf yönetimi (class_weight='balanced')
- Optimal K seçimi (Elbow yöntemi)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn import tree
import warnings
warnings.filterwarnings('ignore')

In [ ]:
data = pd.read_csv('bank_churn.csv')
data

In [ ]:
# Veri seti hakkında genel bilgi
data.info()

In [ ]:
# Sayısal sütunların istatistikleri
data.describe()

In [ ]:
# İlk 5 satır
data.head()

In [ ]:
# Hedef değişkenin dağılımı
print(data['Exited'].value_counts())
print(f"\nChurn oranı: {data['Exited'].mean():.2%}")

## Keşifsel Veri Analizi (EDA)

In [ ]:
# Hedef değişken dağılımı (bar chart)
plt.figure(figsize=(6,4))
data['Exited'].value_counts().plot(kind='bar', color=['#2ecc71','#e74c3c'])
plt.title('Müşteri Churn Dağılımı')
plt.xlabel('Exited (0=Kaldı, 1=Ayrıldı)')
plt.ylabel('Müşteri Sayısı')
plt.xticks(rotation=0)
plt.show()

In [ ]:
# Ülkelere göre churn dağılımı
pd.crosstab(data['Geography'], data['Exited']).plot(kind='bar', figsize=(8,4), color=['#2ecc71','#e74c3c'])
plt.title('Ülkelere Göre Churn Dağılımı')
plt.ylabel('Müşteri Sayısı')
plt.xticks(rotation=0)
plt.legend(['Kaldı','Ayrıldı'])
plt.show()

In [ ]:
# Cinsiyete göre churn dağılımı
pd.crosstab(data['Gender'], data['Exited']).plot(kind='bar', figsize=(8,4), color=['#2ecc71','#e74c3c'])
plt.title('Cinsiyete Göre Churn Dağılımı')
plt.ylabel('Müşteri Sayısı')
plt.xticks(rotation=0)
plt.legend(['Kaldı','Ayrıldı'])
plt.show()

## Veri Ön İşleme

In [ ]:
# Gereksiz sütunları kaldırma (ID, isim gibi)
df = data.drop(['RowNumber', 'CustomerId', 'Surname'], axis=1)
df.head()

In [ ]:
# One-Hot Encoding (nominal kategorik değişkenler için Ordinal Encoding yerine)
df = pd.get_dummies(df, columns=['Geography', 'Gender'], drop_first=True)
df.head()

In [ ]:
# Bağımsız değişkenler (X) ve bağımlı değişken (y)
X = df.drop(['Exited'], axis=1)
y = df['Exited']

print("X shape:", X.shape)
print("y shape:", y.shape)
print("\nÖznitelikler:", list(X.columns))

In [ ]:
# Eğitim ve test setlerine ayırma (%67 eğitim, %33 test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42, stratify=y)

print(f"Eğitim seti: {X_train.shape[0]} kayıt")
print(f"Test seti: {X_test.shape[0]} kayıt")
print(f"\nEğitim churn oranı: {y_train.mean():.2%}")
print(f"Test churn oranı: {y_test.mean():.2%}")

In [ ]:
# Ölçeklendirme (KNN için gerekli - mesafe tabanlı algoritma)
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns, index=X_train.index)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns, index=X_test.index)

print("Ölçeklendirme tamamlandı.")
X_train_scaled.describe().round(2)

## Karar Ağacı Modeli

In [ ]:
# Karar Ağacı - Hiperparametre Optimizasyonu (GridSearchCV)
param_grid_dt = {
    'max_depth': [2, 3, 4, 5, 6, 7, 8],
    'criterion': ['gini', 'entropy'],
    'min_samples_split': [2, 5, 10, 20],
    'min_samples_leaf': [1, 5, 10]
}

grid_dt = GridSearchCV(
    DecisionTreeClassifier(class_weight='balanced', random_state=0),
    param_grid_dt,
    cv=5,
    scoring='f1',
    n_jobs=-1
)
grid_dt.fit(X_train, y_train)

print("En iyi parametreler:", grid_dt.best_params_)
print(f"En iyi CV F1 skoru: {grid_dt.best_score_:.4f}")

In [ ]:
# En iyi Karar Ağacı modeli
KA = grid_dt.best_estimator_
tahmin = KA.predict(X_test)

print(f"Test Accuracy: {accuracy_score(y_test, tahmin):.4f}")

In [ ]:
# Karar Ağacı - 5-Fold Cross Validation
cv_scores_dt = cross_val_score(KA, X_train, y_train, cv=5, scoring='accuracy')
print(f"CV Accuracy Skorları: {cv_scores_dt.round(4)}")
print(f"CV Accuracy Ortalaması: {cv_scores_dt.mean():.4f} (+/- {cv_scores_dt.std():.4f})")

cv_f1_dt = cross_val_score(KA, X_train, y_train, cv=5, scoring='f1')
print(f"\nCV F1 Skorları: {cv_f1_dt.round(4)}")
print(f"CV F1 Ortalaması: {cv_f1_dt.mean():.4f} (+/- {cv_f1_dt.std():.4f})")

In [ ]:
# Karışıklık Matrisi (Confusion Matrix)
cm = confusion_matrix(y_test, tahmin)

plt.figure(figsize=(6,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Kaldı','Ayrıldı'],
            yticklabels=['Kaldı','Ayrıldı'])
plt.title('Karar Ağacı - Confusion Matrix')
plt.xlabel('Tahmin')
plt.ylabel('Gerçek')
plt.show()

In [ ]:
# Sınıflandırma Raporu
print(classification_report(y_test, tahmin, target_names=['Kaldı (0)', 'Ayrıldı (1)']))

## Karar Ağacı Görselleştirme

In [ ]:
# Karar ağacını görselleştirme
plt.figure(figsize=(20,10))
tree.plot_tree(KA,
               feature_names=X_train.columns,
               class_names=['Kaldı','Ayrıldı'],
               filled=True,
               rounded=True,
               fontsize=10)
plt.title(f'Karar Ağacı (max_depth={KA.max_depth}, criterion={KA.criterion})')
plt.tight_layout()
plt.show()

In [ ]:
# Öznitelik Önemlilikleri (Feature Importance)
importance = pd.Series(KA.feature_importances_, index=X_train.columns).sort_values(ascending=True)

plt.figure(figsize=(8,5))
importance.plot(kind='barh', color='steelblue')
plt.title('Karar Ağacı - Öznitelik Önemlilikleri')
plt.xlabel('Önemlilik Skoru')
plt.tight_layout()
plt.show()

## KNN ile Karşılaştırma

In [ ]:
# Optimal K değeri seçimi (Elbow Yöntemi)
k_range = range(1, 21)
k_scores = []

for k in k_range:
    knn_temp = KNeighborsClassifier(n_neighbors=k)
    scores = cross_val_score(knn_temp, X_train_scaled, y_train, cv=5, scoring='accuracy')
    k_scores.append(scores.mean())

plt.figure(figsize=(10,5))
plt.plot(k_range, k_scores, 'bo-')
plt.xlabel('K Değeri')
plt.ylabel('Cross-Validated Accuracy')
plt.title('KNN - Optimal K Seçimi (Elbow Yöntemi)')
plt.xticks(k_range)
plt.grid(True, alpha=0.3)
plt.show()

optimal_k = k_range[np.argmax(k_scores)]
print(f"\nOptimal K: {optimal_k} (Accuracy: {max(k_scores):.4f})")

In [ ]:
# KNN Sınıflandırıcı (ölçeklendirilmiş veri + optimal K)
classifier_knn = KNeighborsClassifier(n_neighbors=optimal_k)
classifier_knn.fit(X_train_scaled, y_train)

tahmin_knn = classifier_knn.predict(X_test_scaled)

print(f"KNN Test Accuracy: {accuracy_score(y_test, tahmin_knn):.4f}")

In [ ]:
# KNN - 5-Fold Cross Validation
cv_scores_knn = cross_val_score(classifier_knn, X_train_scaled, y_train, cv=5, scoring='accuracy')
print(f"CV Accuracy Skorları: {cv_scores_knn.round(4)}")
print(f"CV Accuracy Ortalaması: {cv_scores_knn.mean():.4f} (+/- {cv_scores_knn.std():.4f})")

cv_f1_knn = cross_val_score(classifier_knn, X_train_scaled, y_train, cv=5, scoring='f1')
print(f"\nCV F1 Skorları: {cv_f1_knn.round(4)}")
print(f"CV F1 Ortalaması: {cv_f1_knn.mean():.4f} (+/- {cv_f1_knn.std():.4f})")

In [ ]:
# KNN Karışıklık Matrisi
cm_knn = confusion_matrix(y_test, tahmin_knn)

plt.figure(figsize=(6,4))
sns.heatmap(cm_knn, annot=True, fmt='d', cmap='Oranges',
            xticklabels=['Kaldı','Ayrıldı'],
            yticklabels=['Kaldı','Ayrıldı'])
plt.title('KNN - Confusion Matrix')
plt.xlabel('Tahmin')
plt.ylabel('Gerçek')
plt.show()

In [ ]:
# KNN Sınıflandırma Raporu
print(classification_report(y_test, tahmin_knn, target_names=['Kaldı (0)', 'Ayrıldı (1)']))

## Model Karşılaştırması

In [ ]:
# Model Karşılaştırması
results = pd.DataFrame({
    'Model': ['Karar Ağacı', 'KNN'],
    'Test Accuracy': [accuracy_score(y_test, tahmin), accuracy_score(y_test, tahmin_knn)],
    'CV Accuracy (Ort.)': [cv_scores_dt.mean(), cv_scores_knn.mean()],
    'CV F1 (Ort.)': [cv_f1_dt.mean(), cv_f1_knn.mean()]
})
results = results.round(4)
print(results.to_string(index=False))

In [ ]:
# Görsel Karşılaştırma
fig, axes = plt.subplots(1, 2, figsize=(14,5))

# Confusion matrix karşılaştırması
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Kaldı','Ayrıldı'], yticklabels=['Kaldı','Ayrıldı'])
axes[0].set_title('Karar Ağacı')
axes[0].set_xlabel('Tahmin')
axes[0].set_ylabel('Gerçek')

sns.heatmap(cm_knn, annot=True, fmt='d', cmap='Oranges', ax=axes[1],
            xticklabels=['Kaldı','Ayrıldı'], yticklabels=['Kaldı','Ayrıldı'])
axes[1].set_title('KNN')
axes[1].set_xlabel('Tahmin')
axes[1].set_ylabel('Gerçek')

plt.suptitle('Confusion Matrix Karşılaştırması', fontsize=14)
plt.tight_layout()
plt.show()